In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CANDIDATE_TABLE = "workspace.firmable_v2_silver.manager_candidate_scores"
EMPLOYEE_TABLE = "workspace.firmable_v2_silver.employees_clean"
OUTPUT_TABLE = "workspace.firmable_v2_silver.manager_assignments"

MIN_SCORE = 55
MAX_ROUNDS = 3

candidates = spark.table(CANDIDATE_TABLE)
employees = spark.table(EMPLOYEE_TABLE)

# Confirm the expected scoring output schema before assignment.
required_columns = {
    "company_id",
    "employee_id",
    "candidate_manager_id",
    "candidate_score",
    "candidate_rank",
    "score_margin",
    "ambiguity_flag",
    "manager_seniority"
}

missing_columns = required_columns - set(candidates.columns)

assert not missing_columns, (
    f"Missing required columns in candidate scores: {sorted(missing_columns)}"
)

# Candidate summary is used to explain why an employee was not assigned.
candidate_summary = (
    candidates
    .groupBy("company_id", "employee_id")
    .agg(
        F.count("*").alias("candidate_count"),
        F.max("candidate_score").alias("best_candidate_score"),
        F.max(F.col("ambiguity_flag").cast("int")).alias("is_ambiguous")
    )
)

# Ambiguous employees are excluded from all assignment rounds.
eligible_candidates = (
    candidates
    .filter(
        (F.col("candidate_score") >= MIN_SCORE) &
        (F.col("ambiguity_flag") == False)
    )
)

# Direct-report capacity is a maximum, not a target.
eligible_candidates = eligible_candidates.withColumn(
    "manager_capacity",
    F.when(F.col("manager_seniority") <= 0, 8)
     .when(F.col("manager_seniority") <= 7, 10)
     .when(F.col("manager_seniority") <= 9, 8)
     .otherwise(6)
)

accepted_assignments = None

# Each round considers the next-ranked candidate for employees
# who were not assigned in an earlier round.
for round_number in range(1, MAX_ROUNDS + 1):

    if accepted_assignments is None:
        remaining_employees = employees.select(
            "company_id", "employee_id"
        ).dropDuplicates()
    else:
        assigned_keys = accepted_assignments.select(
            "company_id", "employee_id"
        ).dropDuplicates()

        remaining_employees = (
            employees.select("company_id", "employee_id")
            .dropDuplicates()
            .join(
                assigned_keys,
                ["company_id", "employee_id"],
                "left_anti"
            )
        )

    round_candidates = (
        eligible_candidates
        .filter(F.col("candidate_rank") == round_number)
        .join(
            remaining_employees,
            ["company_id", "employee_id"],
            "inner"
        )
    )

    if accepted_assignments is None:
        manager_load = None
    else:
        manager_load = (
            accepted_assignments
            .groupBy("company_id", "candidate_manager_id")
            .agg(F.count("*").alias("current_direct_reports"))
        )

    if manager_load is not None:
        round_candidates = round_candidates.join(
            manager_load,
            ["company_id", "candidate_manager_id"],
            "left"
        )
    else:
        round_candidates = round_candidates.withColumn(
            "current_direct_reports", F.lit(0)
        )

    round_candidates = (
        round_candidates
        .withColumn(
            "current_direct_reports",
            F.coalesce(F.col("current_direct_reports"), F.lit(0))
        )
        .withColumn(
            "available_capacity",
            F.col("manager_capacity") - F.col("current_direct_reports")
        )
        .filter(F.col("available_capacity") > 0)
    )

    # When several employees compete for the same manager,
    # prioritize the strongest candidate evidence.
    manager_window = (
        Window
        .partitionBy("company_id", "candidate_manager_id")
        .orderBy(
            F.col("candidate_score").desc(),
            F.col("score_margin").desc_nulls_last(),
            F.col("employee_id").asc()
        )
    )

    round_accepted = (
        round_candidates
        .withColumn(
            "manager_slot_rank",
            F.row_number().over(manager_window)
        )
        .filter(
            F.col("manager_slot_rank") <= F.col("available_capacity")
        )
        .withColumn("assignment_round", F.lit(round_number))
        .withColumn(
            "assignment_status",
            F.lit("ASSIGNED")
        )
        .select(
            "company_id",
            "employee_id",
            "candidate_manager_id",
            "candidate_score",
            "candidate_rank",
            "score_margin",
            "relationship_type",
            "manager_seniority",
            "manager_capacity",
            "assignment_round",
            "assignment_status"
        )
    )

    if accepted_assignments is None:
        accepted_assignments = round_accepted
    else:
        accepted_assignments = accepted_assignments.unionByName(
            round_accepted
        )

# Handle the case where no candidate was accepted.
if accepted_assignments is None:
    raise Exception("No manager assignments were generated.")

# Build one final row for every employee, including roots and abstentions.
assignment_output = (
    employees
    .select("company_id", "employee_id")
    .dropDuplicates()
    .join(
        accepted_assignments,
        ["company_id", "employee_id"],
        "left"
    )
    .join(
        candidate_summary,
        ["company_id", "employee_id"],
        "left"
    )
    .withColumn(
        "assignment_status",
        F.when(
            F.col("candidate_manager_id").isNotNull(),
            F.lit("ASSIGNED")
        )
        .when(
            F.col("is_ambiguous") == 1,
            F.lit("ABSTAINED_AMBIGUOUS")
        )
        .when(
            F.col("best_candidate_score").isNotNull() &
            (F.col("best_candidate_score") < MIN_SCORE),
            F.lit("ABSTAINED_LOW_SCORE")
        )
        .when(
            F.col("candidate_count").isNotNull(),
            F.lit("UNASSIGNED_CAPACITY_OR_ELIGIBILITY")
        )
        .otherwise(F.lit("NO_CANDIDATE_OR_ROOT"))
    )
    .drop("is_ambiguous")
)

# Integrity checks.
total_employees = employees.select(
    "company_id", "employee_id"
).dropDuplicates().count()

assigned_count = assignment_output.filter(
    F.col("candidate_manager_id").isNotNull()
).count()

duplicate_assignments = (
    assignment_output
    .groupBy("company_id", "employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

self_relationships = assignment_output.filter(
    F.col("employee_id") == F.col("candidate_manager_id")
).count()



# Check that no manager exceeds their configured direct-report capacity.
capacity_violations = (
    assignment_output
    .filter(F.col("candidate_manager_id").isNotNull())
    .groupBy("company_id", "candidate_manager_id")
    .agg(
        F.count("*").alias("actual_direct_reports"),
        F.max("manager_capacity").alias("manager_capacity")
    )
    .filter(F.col("actual_direct_reports") > F.col("manager_capacity"))
    .count()
)

assert total_employees == assignment_output.count()
assert duplicate_assignments == 0
assert self_relationships == 0
assert capacity_violations == 0

# Validate seniority using the actual employee and manager records.
employee_levels = employees.select(
    "company_id",
    F.col("employee_id").alias("checked_employee_id"),
    F.col("inferred_seniority_level").alias("employee_level")
)

manager_levels = employees.select(
    "company_id",
    F.col("employee_id").alias("checked_manager_id"),
    F.col("inferred_seniority_level").alias("manager_level")
)

seniority_check = (
    assignment_output
    .filter(F.col("candidate_manager_id").isNotNull())
    .join(
        employee_levels,
        (assignment_output.company_id == employee_levels.company_id) &
        (assignment_output.employee_id == employee_levels.checked_employee_id),
        "inner"
    )
    .join(
        manager_levels,
        (assignment_output.company_id == manager_levels.company_id) &
        (assignment_output.candidate_manager_id == manager_levels.checked_manager_id),
        "inner"
    )
)

invalid_seniority = seniority_check.filter(
    F.col("manager_level") >= F.col("employee_level")
).count()

assert invalid_seniority == 0, (
    f"Found {invalid_seniority} assignments violating seniority ordering."
)

# Publish assignments.
(
    assignment_output
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

print("MANAGER ASSIGNMENT COMPLETED")
print(f"Total employees: {total_employees}")
print(f"Assigned employees: {assigned_count}")
print(f"Unassigned employees: {total_employees - assigned_count}")
print(f"Assignment rate: {assigned_count / total_employees * 100:.2f}%")
print(f"Minimum candidate score: {MIN_SCORE}")
print(f"Assignment rounds: {MAX_ROUNDS}")

print("\nASSIGNMENT STATUS")
spark.table(OUTPUT_TABLE).groupBy(
    "assignment_status"
).count().orderBy(
    "assignment_status"
).show(truncate=False)

print("\nASSIGNMENTS BY ROUND")
spark.table(OUTPUT_TABLE).filter(
    F.col("candidate_manager_id").isNotNull()
).groupBy(
    "assignment_round"
).count().orderBy(
    "assignment_round"
).show()

print("\nMANAGER CAPACITY CHECK")
print(f"Capacity violations: {capacity_violations}")
print(f"Duplicate employee assignments: {duplicate_assignments}")
print(f"Self relationships: {self_relationships}")
print(f"Invalid seniority relationships: {invalid_seniority}")

print(f"\nOUTPUT TABLE: {OUTPUT_TABLE}")
print("ALL ASSIGNMENT INTEGRITY CHECKS PASSED")